# SEC EDGAR filings and company identifiers

Inspect Apple's recent 10-K and 10-Q filings, companyfacts response, and XBRL Assets records. Apple's CIK is 320193, padded to 0000320193 in API paths.

Read a fact together with its unit, form, reporting period, filing date and accession number. Companyfacts contains structured values; it does not replace the filing text and financial-statement notes.

This notebook makes live SEC requests when run. Configure SEC_CONTACT_EMAIL locally; the contact value is not saved or printed.


In [ ]:
from pathlib import Path
import os
import time

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display

current_path = Path.cwd().resolve()
project_root = next(
    (
        candidate
        for candidate in (current_path, *current_path.parents)
        if (candidate / "data").is_dir() and (candidate / "src").is_dir()
    ),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run this notebook from within the project directory.")


In [ ]:
SEC_TICKERS_URL = "https://www.sec.gov/files/company_tickers.json"
SEC_SUBMISSIONS_URL_TEMPLATE = (
    "https://data.sec.gov/submissions/CIK{cik}.json"
)
SEC_COMPANYFACTS_URL_TEMPLATE = (
    "https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
)

REQUEST_TIMEOUT_SECONDS = 30
REQUEST_INTERVAL_SECONDS = 0.2

print("request_timeout_seconds:", REQUEST_TIMEOUT_SECONDS)
print("request_interval_seconds:", REQUEST_INTERVAL_SECONDS)

In [ ]:
load_dotenv(project_root / ".env", override=False)
sec_contact_email = os.getenv("SEC_CONTACT_EMAIL", "").strip()
if not sec_contact_email or "@" not in sec_contact_email:
    raise ValueError("Set SEC_CONTACT_EMAIL in the local .env file or environment before running this notebook.")

sec_headers = {
    "User-Agent": f"U.S. Financial Research Project {sec_contact_email}",
    "Accept-Encoding": "gzip, deflate",
}


In [ ]:
def get_sec_json(
        url,
        headers,
        pause_seconds=REQUEST_INTERVAL_SECONDS,
):
    if pause_seconds < 0:
        raise ValueError("pause_seconds must be nonnegative.")

    time.sleep(pause_seconds)

    response = requests.get(
        url,
        headers=headers,
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    payload = response.json()

    if not isinstance(payload, dict):
        raise TypeError(
            "The SEC response must be a JSON object."
        )

    return payload

In [ ]:
ticker_payload = get_sec_json(
    url=SEC_TICKERS_URL,
    headers=sec_headers,
)

ticker_df = pd.DataFrame(ticker_payload.values())

required_ticker_columns = {
    "cik_str",
    "ticker",
    "title",
}
missing_ticker_columns = (
    required_ticker_columns - set(ticker_df.columns)
)

if missing_ticker_columns:
    raise ValueError(
        "The SEC ticker table is missing required columns: "
        f"{sorted(missing_ticker_columns)}"
    )

ticker_df["ticker"] = ticker_df["ticker"].str.upper()

apple_ticker_df = ticker_df.loc[
    ticker_df["ticker"] == "AAPL",
    ["cik_str", "ticker", "title"],
].reset_index(drop=True)

if len(apple_ticker_df) != 1:
    raise ValueError(
        "Could not identify a unique AAPL row "
        "in the response."
    )

print("ticker_row_count:", len(ticker_df))
display(apple_ticker_df)

In [ ]:
apple_cik_int = int(apple_ticker_df.loc[0, "cik_str"])
apple_cik = f"{apple_cik_int:010d}"
expected_apple_cik = "0000320193"

if apple_cik != expected_apple_cik:
    raise ValueError(
        f"Unexpected Apple CIK: {apple_cik}"
    )

print("apple_cik_int:", apple_cik_int)
print("apple_cik:", apple_cik)

In [ ]:
submissions_url = SEC_SUBMISSIONS_URL_TEMPLATE.format(
    cik=apple_cik
)

apple_submissions_json = get_sec_json(
    url=submissions_url,
    headers=sec_headers,
)

if "filings" not in apple_submissions_json:
    raise KeyError("The submissions response has no filings field.")

if "recent" not in apple_submissions_json["filings"]:
    raise KeyError(
        "The submissions response has no filings.recent field."
    )

recent_filings_payload = apple_submissions_json["filings"]["recent"]
recent_filings_df = pd.DataFrame(recent_filings_payload)

required_filing_columns = [
    "filingDate",
    "reportDate",
    "form",
    "accessionNumber",
    "primaryDocument",
]
missing_filing_columns = [
    column
    for column in required_filing_columns
    if column not in recent_filings_df.columns
]

if missing_filing_columns:
    raise ValueError(
        "Recent filings are missing required columns: "
        f"{missing_filing_columns}"
    )

recent_filings_df["filingDate"] = pd.to_datetime(
    recent_filings_df["filingDate"],
    errors="coerce",
)
recent_filings_df["reportDate"] = pd.to_datetime(
    recent_filings_df["reportDate"],
    errors="coerce",
)

periodic_filings_df = (
    recent_filings_df.loc[
        recent_filings_df["form"].isin(["10-K", "10-Q"]),
        required_filing_columns,
    ]
    .sort_values("filingDate", ascending=False)
    .reset_index(drop=True)
)

if periodic_filings_df.empty:
    raise ValueError(
        "No recent 10-K or 10-Q filings were found."
    )

print("submissions_top_level_keys:")
print(sorted(apple_submissions_json.keys()))
print("submissions_name:", apple_submissions_json.get("name"))
print("submissions_cik:", apple_submissions_json.get("cik"))
print("recent_filing_count:", len(recent_filings_df))
print("recent_10k_10q_count:", len(periodic_filings_df))
display(periodic_filings_df.head(10))

In [ ]:
form_count = (
    periodic_filings_df["form"]
    .value_counts()
    .rename_axis("form")
    .reset_index(name="count")
)

print("periodic_form_count:")
display(form_count)

In [ ]:
companyfacts_url = SEC_COMPANYFACTS_URL_TEMPLATE.format(
    cik=apple_cik
)

apple_companyfacts_json = get_sec_json(
    url=companyfacts_url,
    headers=sec_headers,
)

required_companyfacts_keys = {
    "cik",
    "entityName",
    "facts",
}
missing_companyfacts_keys = (
    required_companyfacts_keys
    - set(apple_companyfacts_json.keys())
)

if missing_companyfacts_keys:
    raise ValueError(
        "The companyfacts response is missing required keys: "
        f"{sorted(missing_companyfacts_keys)}"
    )

facts_by_taxonomy = apple_companyfacts_json["facts"]
taxonomy_names = sorted(facts_by_taxonomy.keys())

if "us-gaap" not in facts_by_taxonomy:
    raise KeyError(
        "The companyfacts response has no us-gaap taxonomy."
    )

us_gaap_facts = facts_by_taxonomy["us-gaap"]

print("companyfacts_top_level_keys:")
print(sorted(apple_companyfacts_json.keys()))
print("companyfacts_cik:", apple_companyfacts_json["cik"])
print(
    "companyfacts_entity_name:",
    apple_companyfacts_json["entityName"],
)
print("taxonomy_names:", taxonomy_names)
print("us_gaap_concept_count:", len(us_gaap_facts))

In [ ]:
concept_preview_rows = []

for concept_id in sorted(us_gaap_facts.keys())[:20]:
    concept_data = us_gaap_facts[concept_id]
    concept_preview_rows.append(
        {
            "concept_id": concept_id,
            "label": concept_data.get("label"),
            "unit_names": ", ".join(
                sorted(concept_data.get("units", {}).keys())
            ),
        }
    )

concept_preview_df = pd.DataFrame(concept_preview_rows)

assets_concept_id = "Assets"
assets_concept = us_gaap_facts.get(assets_concept_id)

if assets_concept is None:
    raise KeyError(
        "The us-gaap taxonomy has no Assets concept."
    )

assets_units = assets_concept.get("units", {})
assets_unit_names = sorted(assets_units.keys())

if "USD" not in assets_units:
    raise KeyError("The Assets concept has no USD records.")

print("concept_preview_row_count:", len(concept_preview_df))
display(concept_preview_df)

print("assets_concept_id:", assets_concept_id)
print("assets_label:", assets_concept.get("label"))
print("assets_description:", assets_concept.get("description"))
print("assets_unit_names:", assets_unit_names)

In [ ]:
assets_facts_df = pd.DataFrame(assets_units["USD"])

if assets_facts_df.empty:
    raise ValueError("The Assets concept contains no USD fact records.")

required_assets_columns = {
    "end",
    "val",
    "form",
    "filed",
    "accn",
}
missing_assets_columns = (
    required_assets_columns - set(assets_facts_df.columns)
)

if missing_assets_columns:
    raise ValueError(
        "Assets facts are missing required columns: "
        f"{sorted(missing_assets_columns)}"
    )

assets_facts_df["end"] = pd.to_datetime(
    assets_facts_df["end"],
    errors="coerce",
)
assets_facts_df["filed"] = pd.to_datetime(
    assets_facts_df["filed"],
    errors="coerce",
)
assets_facts_df["val"] = pd.to_numeric(
    assets_facts_df["val"],
    errors="coerce",
)

assets_periodic_df = (
    assets_facts_df.loc[
        assets_facts_df["form"].isin(["10-K", "10-Q"])
    ]
    .sort_values(
        ["end", "filed"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

if assets_periodic_df.empty:
    raise ValueError(
        "No 10-K or 10-Q records were found for Assets."
    )

fact_display_columns = [
    "end",
    "val",
    "form",
    "fy",
    "fp",
    "filed",
    "accn",
    "frame",
]
available_fact_columns = [
    column
    for column in fact_display_columns
    if column in assets_periodic_df.columns
]

repeated_end_mask = assets_periodic_df.duplicated(
    subset=["end"],
    keep=False,
)
repeated_end_df = assets_periodic_df.loc[
    repeated_end_mask,
    available_fact_columns,
].sort_values(
    ["end", "filed"],
    ascending=[False, False],
)

print("assets_fact_count:", len(assets_facts_df))
print("assets_columns:", list(assets_facts_df.columns))
print(
    "assets_missing_end_count:",
    assets_facts_df["end"].isna().sum(),
)
print(
    "assets_missing_val_count:",
    assets_facts_df["val"].isna().sum(),
)
print("assets_10k_10q_count:", len(assets_periodic_df))
print("repeated_end_record_count:", len(repeated_end_df))

display(assets_periodic_df[available_fact_columns].head(15))
display(repeated_end_df.head(20))

In [ ]:
revenue_keywords = ("Revenue", "Sales")
revenue_candidate_ids = sorted(
    concept_id
    for concept_id in us_gaap_facts
    if any(
        keyword in concept_id
        for keyword in revenue_keywords
    )
)

revenue_candidate_rows = []

for concept_id in revenue_candidate_ids:
    concept_data = us_gaap_facts[concept_id]
    revenue_candidate_rows.append(
        {
            "concept_id": concept_id,
            "label": concept_data.get("label"),
            "description": concept_data.get("description"),
            "unit_names": ", ".join(
                sorted(concept_data.get("units", {}).keys())
            ),
        }
    )

revenue_candidate_df = pd.DataFrame(revenue_candidate_rows)

print("revenue_candidate_count:", len(revenue_candidate_df))
display(revenue_candidate_df.head(20))

In [ ]:
apple_structure_summary = {
    "ticker": "AAPL",
    "cik_integer": apple_cik_int,
    "cik_10_digit": apple_cik,
    "entity_name": apple_companyfacts_json["entityName"],
    "taxonomy_names": ", ".join(taxonomy_names),
    "us_gaap_concept_count": len(us_gaap_facts),
    "recent_10k_10q_count": len(periodic_filings_df),
    "assets_unit_names": ", ".join(assets_unit_names),
    "assets_fact_count": len(assets_facts_df),
    "revenue_candidate_count": len(revenue_candidate_df),
}

apple_structure_summary_df = (
    pd.Series(apple_structure_summary, name="value")
    .rename_axis("item")
    .reset_index()
)

display(apple_structure_summary_df)